# 沖縄県の植物観察データの視覚化

## 依存ライブラリインポート 

In [ ]:
from pathlib import Path
import time
import json

from pygbif import occurrences
import pandas as pd
import numpy as np
from tqdm.notebook import tqdm

# GBIF観測データの読み込み

In [ ]:
total_num = 3000
limit = 300
init_offset = 0
# final = init_offset + total_num
final = init_offset + limit*(total_num//limit)
print(f"{init_offset=}, {final=}, {list(range(init_offset,
                         final,
                         limit))=}")

In [ ]:
all_results = []


for offset in tqdm(range(init_offset,
                         final,
                         limit)):
    res = occurrences.search(
        country="JP",
        stateProvince="Okinawa",
        kingdomKey=6,  # Plantae
        hasCoordinate=True,
        year="2015,2026",
        limit=limit,
        offset=offset
    )

    results = res["results"]
    all_results.extend(results)

    # print(f"offset={offset}, got={len(results)}, total_so_far={len(all_results)}")

    time.sleep(0.2)  # 念のため軽く待つ

df = pd.DataFrame(all_results)

output_dir = Path("../data/raw")
fname = "gbif_okinawa_plants.parquet"

df.to_parquet(
    output_dir / fname,
    index=False
)

print(f"{df.shape=}, {len(df)=}")
df.head()

In [ ]:
import plotly.express as px

fig = px.scatter_map(
    df,
    lat="decimalLatitude",
    lon="decimalLongitude",
    hover_name="scientificName",
    zoom=5,
    height=600
)

fig.update_layout(mapbox_style="open-street-map")
fig.show()

点データを GeoDataFrame 化

In [ ]:
import geopandas as gpd
from shapely.geometry import Point


geometry = [
    Point(xy) for xy in zip(df.decimalLongitude, df.decimalLatitude)
]

gdf = gpd.GeoDataFrame(
    df,
    geometry=geometry,
    crs="EPSG:4326"
)

## GBIF植物観測頻度ヒートマップ

In [ ]:
center = {"lat": 26.5, "lon": 127.9}

fig = px.density_map(
    df,
    lat="decimalLatitude",
    lon="decimalLongitude",
    radius=20,
    zoom=6,
    center=center,
    map_style="open-street-map",
    height=700
)

fig.update_layout(
    title="GBIF植物観測頻度ヒートマップ"
)

fig.show()

# 市区町村別階級区分図

## 沖縄の市町村データの読みこみ

In [ ]:
import zipfile

zip_path = Path("../data/raw/N03-20250101_47_GML.zip")
out_dir = Path("../data/raw/okinawa_boundary")

with zipfile.ZipFile(zip_path, "r") as z:
    z.extractall(out_dir)

list(out_dir.iterdir())[:10]

In [ ]:
shp_files = list(out_dir.rglob("*.shp"))
shp_files

In [ ]:
municipalities = gpd.read_file(shp_files[0])
municipalities.head(10)

In [ ]:
municipalities.columns

国土数値情報の行政区域データでは、だいたい以下のような列があります。

- N03_001  都道府県名
- N03_002  支庁・振興局名など
- N03_003  郡・政令市名など
- N03_004  市区町村名
- N03_007  行政区域コード
- geometry  境界ポリゴン

In [ ]:
okinawa_muni = municipalities[municipalities["N03_001"] == "沖縄県"].copy()
okinawa_muni[["N03_001", "N03_004", "N03_007"]].head()

In [ ]:
#print(f"沖縄県内の市区町村の数 + 1 ={okinawa_muni['N03_004'].nunique()}")
print(f"ポリゴンの数={len(okinawa_muni)}")

In [ ]:
okinawa_muni.plot(figsize=(8, 8))

In [ ]:
okinawa_muni = okinawa_muni.to_crs("EPSG:4326")

## 各指標の計算

### Species Richness

In [ ]:
joined = gpd.sjoin(
    gdf,
    okinawa_muni[["N03_004", "N03_007", "geometry"]],
    how="inner",
    predicate="within"
)
print(f"{len(joined)=}")
joined[["scientificName", "N03_004"]].head()

In [ ]:
joined.head(3)

In [ ]:

def shannon_entropy(counts):
    """
    counts: 各種の出現数
    """
    proportions = counts / counts.sum()

    return -np.sum(
        proportions * np.log(proportions)
    )


def pielou_evenness(counts):
    H = shannon_entropy(counts)

    S = len(counts)

    if S <= 1:
        return 0

    return H / np.log(S)


def hill_q1(counts):
    """
    counts: 各種の出現数
    """
    return np.exp(shannon_entropy(counts))


def apply_index(df_, _func, _name):
    if _name == "species_richness":
        return (
            df_
            .groupby("N03_004")["species"]
            .nunique()
            .reset_index(name="species_richness")
        )
    else:
        return (
            df_
            .groupby("N03_004")["species"]
            .value_counts()
            .groupby(level=0)
            .apply(_func)
            .reset_index(name=_name)
        )

diversity_indices = {
    "species_richness": None,
    "shannon_entropy": shannon_entropy,
    "pielou_evenness": pielou_evenness,
    "hill_q1": hill_q1
}

In [ ]:
richness = okinawa_muni
for k, func in diversity_indices.items():
    index_df = apply_index(joined, func, k)
    richness = richness.merge(
        index_df,
        on="N03_004",
        how="left"
    )
    richness[k] = richness[k].fillna(0)

In [ ]:
richness

In [ ]:
(richness
    .sort_values('species_richness', ascending=False)
    .drop("geometry", axis=1)
    .drop_duplicates()
    .head(10)
)

In [ ]:
def draw_map(index_map,
            color_target,
            geojson,
            labels,
            center={"lat": 26.5, "lon": 127.9},
            color_scale="Viridis",
            ):
    return px.choropleth_map(
        index_map,
        geojson=geojson,
        locations="N03_004",
        hover_name="N03_004",
        hover_data={color_target: True},    
        color=color_target,
        map_style="open-street-map",
        center=center,
        zoom=6,
        opacity=0.7,
        color_continuous_scale=color_scale,
        labels=labels
    )

In [ ]:
# Plotlyで扱いやすいように GeoJSON 化
geojson = json.loads(richness.to_json())

# GeoJSON内のidを市町村名にする
for feature in geojson["features"]:
    feature["id"] = feature["properties"]["N03_004"]

fig = draw_map(
    index_map=richness,
    color_target="species_richness",
    geojson=geojson,
    labels={"species_richness": "植物種数"}
)

fig.update_layout(
    title="GBIFデータに基づく沖縄県市町村別の植物種数",
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig.show()

### Shannon entropy

In [ ]:
fig = draw_map(
    index_map=richness,
    color_target="shannon_entropy",
    geojson=geojson,
    labels={"shannon_entropy": "Shannon Entropy"},    
)

fig.update_layout(
    title="沖縄県市町村別 Shannon Entropy",
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig.show()

### Pielou evenness

In [ ]:
fig = draw_map(
    index_map=richness,
    color_target="pielou_evenness",
    geojson=geojson,
    labels={"pielou_evenness": "Pielou Evenness"}
)

fig.update_layout(
    title="沖縄県市町村別 Pielou Evenness",
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig.show()

### Hill number(q=1)

In [ ]:
fig = draw_map(
    index_map=richness,
    color_target="hill_q1",
    geojson=geojson,
    labels={"hill_q1": "Hill Number(q=1)"}
)
fig.update_layout(
    title="沖縄県市町村別 Hill Number(q=1)",
    margin={"r":0, "t":50, "l":0, "b":0}
)

fig.show()